### Extração de dados Inter

In [1]:
import pdfplumber
import pandas as pd
import re
import numpy as np
import re
from datetime import datetime


In [ ]:
# escolha se é PE ou Exposição Bruta:
# PE = d. Análise da movimentação das perdas esperadas por estágio: (pág 42)
# Exposição Bruta = c. Análise da movimentação dos empréstimos e adiantamentos a clientes por estágio (pág 41)

In [2]:
with pdfplumber.open("Demonstrações Financeiras IFRS 3T25.pdf") as pdf:
    page = pdf.pages[41]   # página 42 (índice começa em 0)
    text = page.extract_text()

# Remove espaços em branco extras de cada linha ( pré-processamento )
text = "\n".join(line.strip() for line in text.splitlines())

# Define início ( start ) e fim ( end ) do trecho a ser extraído. 
start = text.find("c. Análise da movimentação dos empréstimos e adiantamentos a clientes por estágio")
end = text.find("Consolidado", start) # se pegar só total, conseguimos apenas o primeiro estágio. 
    
trecho = text[start:end]
print(trecho)

# divide em linhas para facilitar a criação do dataframe
linhas = [l.strip() for l in trecho.splitlines() if l.strip()] 
    

c. Análise da movimentação dos empréstimos e adiantamentos a clientes por estágio
Saldo inicial em Transferência para Transferência para Transferência do Transferência do Contratos Originação / Saldo final em Saldo final em
Estágio 1 Baixas para prejuízo
01/01/2025 estágio 2 estágio 3 estágio 2 estágio 3 finalizados (recebimento) 30/09/2025 31/12/2024
Cartão de crédito 10.330.639 (1.616.314) (2.938) 844.335 9 (2.845.865) — 5.467.227 12.177.093 10.330.639
Imobiliário 10.196.928 (2.044.376) (17.754) 1.465.608 15.094 (965.913) — 4.565.030 13.214.617 10.196.928
Pessoal 7.389.879 (611.285) (61.959) 287.621 243.736 (1.796.687) — 4.714.333 10.165.638 7.389.879
Empresas 3.887.678 (189.960) (6.484) 70.382 — (5.573.692) — 5.627.673 3.815.596 3.887.678
Rural 340.834 (8.798) (743) — — (229.102) — 230.961 333.152 340.834
Total 32.145.958 (4.470.733) (89.878) 2.667.946 258.839 (11.411.259) — 20.605.224 39.706.096 32.145.958
Saldo inicial em Transferência para Transferência para Transferência do Tran

In [3]:
PRODUTOS = [
    "Cartão de crédito",
    "Imobiliário",
    "Pessoal",
    "Empresas",
    "Rural",
]

def eh_produto(linha: str) -> bool:
    return any(linha.startswith(p) for p in PRODUTOS)

def reduzir_linha_produto(linha: str) -> str:
    """
    Mantém:
    - nome do produto
    - os dois últimos elementos da linha (número ou —)
    """
    partes = linha.split()

    # últimos dois elementos SEMPRE preservados
    ultimos_dois = partes[-2:]

    # nome do produto pode ter mais de uma palavra
    for produto in PRODUTOS:
        if linha.startswith(produto):
            return f"{produto} {' '.join(ultimos_dois)}"

    return linha  # fallback (não deveria acontecer)

def reduzir_texto(texto: str) -> str:
    linhas = [l.strip() for l in texto.splitlines() if l.strip()]
    saida = []

    for linha in linhas:
        # Regra 2
        if linha.startswith("Total"):
            continue

        # Regra 3
        if linha.lower().startswith("saldo inicial"):
            continue

        # Regra 4 + 1
        if eh_produto(linha):
            saida.append(reduzir_linha_produto(linha))
        else:
            # Mantém cabeçalhos, títulos, estágios, datas etc
            saida.append(linha)

    return "\n".join(saida)


In [4]:
texto_reduzido = reduzir_texto(trecho)
print(texto_reduzido)


c. Análise da movimentação dos empréstimos e adiantamentos a clientes por estágio
Estágio 1 Baixas para prejuízo
01/01/2025 estágio 2 estágio 3 estágio 2 estágio 3 finalizados (recebimento) 30/09/2025 31/12/2024
Cartão de crédito 12.177.093 10.330.639
Imobiliário 13.214.617 10.196.928
Pessoal 10.165.638 7.389.879
Empresas 3.815.596 3.887.678
Rural 333.152 340.834
Estágio 2 Baixas para prejuízo
01/01/2025 estágio 1 estágio 3 estágio 1 estágio 3 finalizados (recebimento) 30/09/2025 31/12/2024
Cartão de crédito 517.743 281.503
Imobiliário 782.884 835.131
Pessoal 226.489 257.816
Empresas 41.098 44.090
Rural 1 —
Estágio 3 Baixas para prejuízo
01/01/2025 estágio 1 estágio 2 estágio 1 estágio 2 finalizados (recebimento) 30/09/2025 31/12/2024
Cartão de crédito 1.272.632 1.187.748
Imobiliário 526.759 218.128
Pessoal 678.432 589.096
Empresas — 36.823
Rural 5.790 —


In [5]:
# Extração de anos e meses

def trimestre_from_date(date_str: str) -> str:
    dt = datetime.strptime(date_str, "%d/%m/%Y")
    trimestre_map = {3: "1T", 6: "2T", 9: "3T", 12: "4T"}
    trimestre = trimestre_map.get(dt.month)

    if not trimestre:
        raise ValueError(f"Mês inesperado na data: {date_str}")

    return f"{trimestre}{str(dt.year)[-2:]}"

def extract_anos(texto: str) -> list[str]:
    datas = re.findall(r"\d{2}/\d{2}/\d{4}", texto)

    if len(datas) < 2:
        raise ValueError("Não foi possível encontrar duas datas finais")

    datas_finais = datas[-2:]
    return [trimestre_from_date(d) for d in datas_finais]


In [6]:
extract_anos(texto_reduzido)
# ['3T25', '4T24']


['3T25', '4T24']

In [7]:
# Separa em blocos

def split_blocos_estagio(texto: str) -> list[tuple[int, list[str]]]:
    linhas = texto.splitlines()

    blocos = []
    estagio_atual = None
    buffer = []

    for linha in linhas:
        m = re.match(r"Estágio (\d) Baixas para prejuízo", linha)
        if m:
            if estagio_atual is not None:
                blocos.append((estagio_atual, buffer))
                buffer = []
            estagio_atual = int(m.group(1))
            continue

        if estagio_atual is not None:
            buffer.append(linha)

    if estagio_atual is not None:
        blocos.append((estagio_atual, buffer))

    return blocos


In [8]:
split_blocos_estagio(texto_reduzido)


[(1,
  ['01/01/2025 estágio 2 estágio 3 estágio 2 estágio 3 finalizados (recebimento) 30/09/2025 31/12/2024',
   'Cartão de crédito 12.177.093 10.330.639',
   'Imobiliário 13.214.617 10.196.928',
   'Pessoal 10.165.638 7.389.879',
   'Empresas 3.815.596 3.887.678',
   'Rural 333.152 340.834']),
 (2,
  ['01/01/2025 estágio 1 estágio 3 estágio 1 estágio 3 finalizados (recebimento) 30/09/2025 31/12/2024',
   'Cartão de crédito 517.743 281.503',
   'Imobiliário 782.884 835.131',
   'Pessoal 226.489 257.816',
   'Empresas 41.098 44.090',
   'Rural 1 —']),
 (3,
  ['01/01/2025 estágio 1 estágio 2 estágio 1 estágio 2 finalizados (recebimento) 30/09/2025 31/12/2024',
   'Cartão de crédito 1.272.632 1.187.748',
   'Imobiliário 526.759 218.128',
   'Pessoal 678.432 589.096',
   'Empresas — 36.823',
   'Rural 5.790 —'])]

In [9]:
def parse_linhas_produto(linhas: list[str]) -> list[dict]:
    rows = []

    for linha in linhas:
        for produto in PRODUTOS:
            if linha.startswith(produto):
                partes = linha.replace(produto, "").strip().split()
                v1, v2 = partes

                rows.append({
                    "produto": produto,
                    "v1": v1,
                    "v2": v2,
                })

    return rows


In [ ]:

def normalizar_valor(valor: str):
    if valor == "—":
        return None

    # remove espaços
    valor = valor.strip()

    # caso (123) → -123
    if re.fullmatch(r"\(\d+\)", valor):
        valor = "-" + valor[1:-1]

    # remove separador de milhar
    valor = valor.replace(".", "")

    return float(valor)

In [11]:
def construir_linhas(texto_reduzido: str) -> list[dict]:
    anos = extract_anos(texto_reduzido)
    blocos = split_blocos_estagio(texto_reduzido)

    linhas_finais = []

    for estagio, linhas_bloco in blocos:
        produtos = parse_linhas_produto(linhas_bloco)

        for ano, chave in zip(anos, ["v1", "v2"]):
            for p in produtos:
                linhas_finais.append({
                    "ano": ano,
                    "banco": "Inter",
                    "produto": p["produto"],
                    "PD": "-",
                    "Estágio": estagio,
                    "Exposição Bruta": normalizar_valor(p[chave]),
                })

    return linhas_finais


In [14]:
df = pd.DataFrame(construir_linhas(texto_reduzido))
df

,ano,banco,produto,PD,Estágio,Exposição Bruta
0,3T25,Inter,Cartão de crédito,-,1,12177093.0
1,3T25,Inter,Imobiliário,-,1,13214617.0
2,3T25,Inter,Pessoal,-,1,10165638.0
3,3T25,Inter,Empresas,-,1,3815596.0
4,3T25,Inter,Rural,-,1,333152.0
5,4T24,Inter,Cartão de crédito,-,1,10330639.0
6,4T24,Inter,Imobiliário,-,1,10196928.0
7,4T24,Inter,Pessoal,-,1,7389879.0
8,4T24,Inter,Empresas,-,1,3887678.0
9,4T24,Inter,Rural,-,1,340834.0
